In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import Binarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# =========================
# 1) Load Dataset
# =========================
df = pd.read_csv('train.csv')[['Age','Fare','SibSp','Parch','Survived']]
df.dropna(inplace=True)

print("✅ Shape after dropna:", df.shape)
print(df.head())


✅ Shape after dropna: (714, 5)
    Age     Fare  SibSp  Parch  Survived
0  22.0   7.2500      1      0         0
1  38.0  71.2833      1      0         1
2  26.0   7.9250      0      0         1
3  35.0  53.1000      1      0         1
4  35.0   8.0500      0      0         0


In [3]:
# =========================
# 2) Feature Engineering: family = SibSp + Parch
# =========================
df['family'] = df['SibSp'] + df['Parch']

# Drop old columns
df.drop(columns=['SibSp','Parch'], inplace=True)

print("\n✅ After creating family feature:")
print(df.head())


✅ After creating family feature:
    Age     Fare  Survived  family
0  22.0   7.2500         0       1
1  38.0  71.2833         1       1
2  26.0   7.9250         1       0
3  35.0  53.1000         1       1
4  35.0   8.0500         0       0


In [7]:
# 3) Split X and y
# =========================
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("\n✅ Train shape:", X_train.shape, "Test shape:", X_test.shape)



✅ Train shape: (571, 3) Test shape: (143, 3)


In [8]:
# 4) WITHOUT Binarization (Baseline)
# ==========================================================
baseline_model = DecisionTreeClassifier(random_state=42)

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

print("\n✅ Accuracy WITHOUT binarization:", accuracy_score(y_test, y_pred))

baseline_cv = np.mean(
    cross_val_score(
        DecisionTreeClassifier(random_state=42),
        X, y,
        cv=10,
        scoring='accuracy'
    )
)
print("✅ CV Accuracy WITHOUT binarization:", baseline_cv)




✅ Accuracy WITHOUT binarization: 0.6223776223776224
✅ CV Accuracy WITHOUT binarization: 0.6429577464788734


In [9]:
# 5) WITH Binarization (ONLY on family column)
# ==========================================================
# Binarizer rule:
# family > 0  => 1 (has family onboard)
# family <= 0 => 0 (alone)
# threshold default = 0.0 (so perfect for this use case)

preprocess = ColumnTransformer(
    transformers=[
        ('bin_family', Binarizer(threshold=0.0, copy=True), ['family'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# Pipeline = preprocessing + model together (best practice)
binarized_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', DecisionTreeClassifier(random_state=42))
])

# Train on train set
binarized_model.fit(X_train, y_train)
y_pred2 = binarized_model.predict(X_test)

print("\n✅ Accuracy WITH binarization:", accuracy_score(y_test, y_pred2))

# CV on full dataset (NO leakage because pipeline handles fit inside CV)
binarized_cv = np.mean(
    cross_val_score(
        binarized_model,
        X, y,
        cv=10,
        scoring='accuracy'
    )
)

print("✅ CV Accuracy WITH binarization:", binarized_cv)



✅ Accuracy WITH binarization: 0.6433566433566433
✅ CV Accuracy WITH binarization: 0.6205790297339593


In [10]:
# 6) Show transformed data sample (for understanding)
# =========================
X_train_trf = preprocess.fit_transform(X_train)

# Since remainder='passthrough', order becomes:
# family (binarized) + other columns (Age, Fare) in same order as X
X_train_trf_df = pd.DataFrame(X_train_trf, columns=['family', 'Age', 'Fare'])

print("\n✅ Transformed Sample (first 5 rows):")
print(X_train_trf_df.head())


✅ Transformed Sample (first 5 rows):
   family   Age     Fare
0     1.0  31.0  20.5250
1     1.0  26.0  14.4542
2     1.0  30.0  16.1000
3     0.0  33.0   7.7750
4     0.0  25.0  13.0000
